# Off-diagonal contraction approximations

Compare all contractions with omission of BWZ and retention of only the two current-enhanced topologies. All changes act only on off-diagonal entries of the reduced GEVP numerator. Inputs are the compact `processData_codex` pickles and current publication eigenvector pickles; no raw HDF5 files are read.

## Setup


In [2]:
from pathlib import Path
import gc, json, os, sys
import numpy as np
import matplotlib.pyplot as plt

PROJECT = Path.cwd().resolve()
assert (PROJECT/"util.py").is_file()
sys.path.insert(0,str(PROJECT))
import util as utils
import util_codex as yuc

OUT = PROJECT/"__codex_ignore/fig/analysis_3pt_topologies_codex/internal_ignore"
OUT.mkdir(parents=True,exist_ok=True)
os.chdir(OUT)
ENSEMBLES = [
    ("cA211.530.24","a24","A24",),
    ("cA2.09.48","a","A48",),
    ("cB211.072.64","b","B64",),
]


## Pickle inputs

Each processing notebook verifies full matrices against the production pickle before saving. Validate the input identity and retain the original ensemble blocking and vacuum subtractions.


In [3]:
from util_codex import OPS, file_sha256


def read_contractions(ensemble, key):
    folder = PROJECT / ensemble
    compact = PROJECT / "__codex_ignore" / ensemble / "pkl/processData_codex/reg_ignore/data_topologies.pkl"
    data = utils.load_pkl(compact)
    if data is None:
        raise FileNotFoundError(f"Run {folder / 'processData_codex.ipynb'} first")
    metadata = data["metadata"]
    reference = folder / "pkl/processData/reg_ignore" / ("data_cd.pkl" if key == "b" else "data.pkl")
    assert metadata["schema"] == 1 and metadata["ensemble"] == ensemble
    assert metadata["operators"] == OPS
    assert file_sha256(reference) == metadata["reference_sha256"], "Production input changed; reprocess topology data"
    assert sorted(data["c3"]) == metadata["separations"]
    return data


## Analysis

Keep the standard NJN entry and the two-point projection fixed. Compute paired differences before estimating their errors.

In [4]:
comparisons, full_ratios = {}, {}
for ensemble,key,label in ENSEMBLES:
    data = read_contractions(ensemble, key)
    c2,c3,matched = (data[name] for name in ["c2", "c3", "matched_c2"])
    if key=="b":
        v,_,w = utils.load_pkl(PROJECT/"__codex_ignore/cB211.072.64/pkl/analysis_2pt_codex/reg_ignore/evec_ratios.pkl")
    else:
        references = utils.load_pkl(PROJECT/"__codex_ignore"/ensemble/
                                    "pkl/analysis_2pt_codex/reg_ignore/two_point_references.pkl")
        v,w = (references["weights"][name] for name in ["v", "w"])
    assert len(v)==len(w)==len(c2)
    mass = utils.ens2amul_iso[key] if key=="b" else utils.ens2amul[key]
    if key=="a":
        mass *= (135/utils.ens2mpi[key])**2
    unit = mass*utils.ens2aInv[key]
    differences = {"no_bwz":{},"direct_only":{}}
    full_ratios[label] = {}
    for tf in sorted(c3):
        full = np.real(c3[tf])
        bwz = np.real(data["bwz"][tf])
        direct = np.real(data["direct"][tf])
        for subset in [bwz,direct]:
            assert np.count_nonzero(subset[:,:,0,0])==0
            np.testing.assert_allclose(subset[:,:,0,1],subset[:,::-1,1,0],rtol=1e-10,atol=1e-12)
        denominator = matched[tf]+v*(c2[:,tf,0,1]+c2[:,tf,1,0])+v**2*c2[:,tf,1,1]
        weight = (v*(1+w))[:,None]
        numerator = (1-w[:,None]**2)*full[:,:,0,0]+weight*(full[:,:,0,1]+full[:,:,1,0])
        full_ratios[label][tf] = numerator/denominator[:,None]*unit
        for case,removed in [("no_bwz",bwz),("direct_only",full-direct)]:
            differences[case][tf] = weight*(removed[:,:,0,1]+removed[:,:,1,0])/denominator[:,None]*unit
    comparisons[label] = {case:utils.symmetrizeRatio(ratios) for case,ratios in differences.items()}
    full_ratios[label] = utils.symmetrizeRatio(full_ratios[label])
    for case,ratios in comparisons[label].items():
        for tf,ratio in ratios.items():
            np.testing.assert_allclose(ratio,ratio[:,::-1],rtol=1e-10,atol=1e-10)
        print(label,case,{tf:utils.jackme_un2str(r[:,tf//2]) for tf,r in ratios.items()},flush=True)
    del data,c2,c3,full,bwz,direct
    gc.collect()


A24 no_bwz {10: '12.9(1.7)', 12: '13.3(2.0)', 14: '12.0(2.3)'}


A24 direct_only {10: '12.8(1.8)', 12: '14.5(2.2)', 14: '14.9(2.8)'}


A48 no_bwz {10: '3.01(87)', 12: '0.7(1.4)', 14: '2.9(2.7)', 16: '1.2(2.8)', 18: '-2.3(3.8)'}


A48 direct_only {10: '2.25(85)', 12: '-0.1(1.4)', 14: '2.0(2.7)', 16: '1.0(3.0)', 18: '-0.2(4.0)'}


B64 no_bwz {8: '0.48(15)', 10: '0.11(24)', 12: '-0.45(45)', 14: '0.76(71)', 16: '0.39(64)', 18: '1.1(1.0)', 20: '3.3(1.8)'}


B64 direct_only {8: '0.18(22)', 10: '-0.13(31)', 12: '-0.88(53)', 14: '0.72(77)', 16: '0.40(84)', 18: '0.8(1.3)', 20: '5.1(2.3)'}


## Figure

Midpoints in physical units. Left: all minus no BWZ. Right: all minus direct only. Horizontal offsets distinguish ensembles.

In [5]:
yuc.apply_paper_style()
fig,axes = plt.subplots(1,2,figsize=(3.4,2.2),sharex=True,sharey=True)
styles = [
    ("a24","A24",utils.colors16[3],"s",-.025),
    ("a","A48",utils.colors16[2],"^",0),
    ("b","B64",utils.colors16[0],"o",.025),
]
rows = {}
for ax,case in zip(axes,["no_bwz","direct_only"]):
    rows[case] = {}
    for key,label,color,marker,shift in styles:
        ratios = comparisons[label][case]
        tfs = sorted(ratios)
        times = np.array(tfs)*utils.ens2a[key]
        midpoint = np.stack([ratios[tf][:,tf//2] for tf in tfs],axis=1)
        mean,error = utils.jackme(midpoint)
        ax.errorbar(times+shift,mean,yerr=error,fmt=marker,color=color,ms=3.5,label=label)
        rows[case][label] = [
            {"ts_a":tf,"ts_fm":float(t),"mean_MeV":float(m),"error_MeV":float(e)}
            for tf,t,m,e in zip(tfs,times,mean,error)]
    utils.addRefLine(ax,0)
    ax.set(xlabel=r"$t_s$ [fm]",xlim=(.55,1.8),xticks=np.arange(.8,1.61,.4),
           ylim=(-10,20),yticks=np.arange(-10,21,5))
axes[0].set_ylabel(r"$\Delta R_{\rm GEVP}^{d}$ [MeV]")
fig.legend(*axes[0].get_legend_handles_labels(),loc="upper center",ncols=3,
           fontsize=7,columnspacing=1,bbox_to_anchor=(.56,1.0))
yuc.finish_shared_y_panels(fig,axes,wspace=.08,rect=(0,0,1,.91))
for suffix in ["png","pdf"]:
    fig.savefig(OUT/("gevp_midpoint_differences."+suffix),dpi=200)
plt.close(fig)
(OUT/"gevp_midpoint_differences.json").write_text(json.dumps(rows,indent=2))
for tf in sorted(full_ratios["A24"]):
    total = full_ratios["A24"][tf][:,tf//2]
    relative = [utils.jackme_un2str(100*comparisons["A24"][case][tf][:,tf//2]/total)
                for case in ["no_bwz","direct_only"]]
    print("A24",tf,"full [MeV]",utils.jackme_un2str(total),"relative shifts [%]",relative)


A24 10 full [MeV] 188.5(5.6) relative shifts [%] ['6.82(74)', '6.80(78)']
A24 12 full [MeV] 187.1(4.9) relative shifts [%] ['7.09(91)', '7.7(1.0)']
A24 14 full [MeV] 186.3(5.9) relative shifts [%] ['6.4(1.1)', '8.0(1.4)']
